# Condensed Matter Simulation

This notebook demonstrates how to simulate lattice spin models using the
`qdk_pythonic.domains.condensed_matter` module. The workflow is:

1. Define a **lattice geometry** (1D chain, 2D square lattice, etc.)
2. Attach a **spin model** (Ising, Heisenberg) to get a Pauli Hamiltonian
3. Build a **Trotterized time-evolution** circuit
4. Analyze gate counts, depth, and resource estimates

Trotter evolution approximates the unitary exp(-iHt) by decomposing H into
individually simulable terms and applying them in small time steps.

In [ ]:
from qdk_pythonic.domains.condensed_matter import (
    Chain, SquareLattice, IsingModel, HeisenbergModel, simulate_dynamics,
)
from qdk_pythonic.domains.common import TrotterEvolution

## 1D Ising Model

The transverse-field Ising model is one of the simplest interacting quantum
spin systems:

$$H = -J \sum_{\langle i,j \rangle} Z_i Z_j \;-\; h \sum_i X_i$$

We define the lattice geometry first, then attach the model to produce a
Pauli Hamiltonian.

In [ ]:
lattice = Chain(8)
model = IsingModel(lattice, J=1.0, h=0.5)
hamiltonian = model.to_hamiltonian()

print(f"Lattice sites: {lattice.num_sites}")
print(f"Nearest-neighbor edges: {len(lattice.edges)}")
print(f"Hamiltonian terms: {len(hamiltonian)}")
print(f"Qubits required: {hamiltonian.qubit_count()}")

## Trotter Evolution

`TrotterEvolution` wraps a Hamiltonian and builds a circuit that approximates
exp(-iHt). More Trotter steps improve accuracy at the cost of deeper circuits.

In [ ]:
evo = TrotterEvolution(hamiltonian=hamiltonian, time=1.0, steps=10)
circuit = evo.to_circuit()

print(f"Gate count: {circuit.gate_count()}")
print(f"Total gates: {circuit.total_gate_count()}")
print(f"Depth: {circuit.depth()}")
print()
print(circuit.draw())

## Scaling Analysis

How does circuit size grow with the number of lattice sites? The ZZ terms scale
linearly with chain length, so we expect near-linear growth in gate count.

In [ ]:
print(f"{'Qubits':>8}  {'Terms':>8}  {'Gates':>8}  {'Depth':>8}")
print("-" * 38)

for n in [4, 8, 12, 16]:
    lat = Chain(n)
    mod = IsingModel(lat, J=1.0, h=0.5)
    ham = mod.to_hamiltonian()
    ev = TrotterEvolution(hamiltonian=ham, time=1.0, steps=10)
    circ = ev.to_circuit()
    print(f"{n:>8}  {len(ham):>8}  {circ.total_gate_count():>8}  {circ.depth():>8}")

## 2D Square Lattice

Moving from 1D to 2D increases the number of nearest-neighbor interactions
significantly. A 3x3 lattice has 9 sites but 12 edges (vs. 8 edges for a
chain of 9).

In [ ]:
lattice_2d = SquareLattice(3, 3)
model_2d = IsingModel(lattice_2d, J=1.0, h=0.5)
ham_2d = model_2d.to_hamiltonian()

evo_2d = TrotterEvolution(hamiltonian=ham_2d, time=1.0, steps=10)
circ_2d = evo_2d.to_circuit()

print(f"Lattice: {lattice_2d.rows}x{lattice_2d.cols} = {lattice_2d.num_sites} sites")
print(f"Edges: {len(lattice_2d.edges)}")
print(f"Hamiltonian terms: {len(ham_2d)}")
print(f"Total gates: {circ_2d.total_gate_count()}")
print(f"Depth: {circ_2d.depth()}")

## Heisenberg Model

The Heisenberg XXZ model includes XX, YY, and ZZ interactions:

$$H = \sum_{\langle i,j \rangle} \left[ J_x X_i X_j + J_y Y_i Y_j + J_z Z_i Z_j \right]$$

It has roughly 3x the Hamiltonian terms of the Ising model on the same lattice,
since each edge contributes three Pauli terms instead of one.

In [ ]:
lattice_heis = Chain(6)
heis_model = HeisenbergModel(lattice_heis, Jx=1.0, Jy=1.0, Jz=1.0)
ham_heis = heis_model.to_hamiltonian()

evo_heis = TrotterEvolution(hamiltonian=ham_heis, time=1.0, steps=10)
circ_heis = evo_heis.to_circuit()

# Compare to Ising on the same lattice
ising_model = IsingModel(lattice_heis, J=1.0, h=0.5)
circ_ising = TrotterEvolution(
    hamiltonian=ising_model.to_hamiltonian(), time=1.0, steps=10,
).to_circuit()

print(f"{'Model':<14} {'Terms':>8} {'Gates':>8} {'Depth':>8}")
print("-" * 42)
print(f"{'Ising':<14} {len(ising_model.to_hamiltonian()):>8} "
      f"{circ_ising.total_gate_count():>8} {circ_ising.depth():>8}")
print(f"{'Heisenberg':<14} {len(ham_heis):>8} "
      f"{circ_heis.total_gate_count():>8} {circ_heis.depth():>8}")

## Resource Estimation

> **Note:** The cell below requires `qsharp>=1.25` (`pip install qsharp>=1.25`).

Resource estimation tells you how many physical qubits and how much wall-clock
time a fault-tolerant quantum computer would need for this simulation.

In [ ]:
# Requires: pip install qsharp>=1.25
# result = evo.estimate_resources()
# print(result)